# MNIST Image Training

Train a TN (model2) to approximate another TN (model1) initialized from an MNIST digit.

- **Loss**: MSE between model2 and model1 contracted tensors
- **Optimizer**: Adam
- **Validation**: side-by-side image comparison (target vs learned vs difference)

In [ ]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()))

%matplotlib inline
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets, transforms

from tneq_qc import QCTN, EngineCommon, BackendFactory, create_optimizer
from tneq_qc.core.tn_tensor import TNTensor


## Configuration

In [ ]:
N_QUBITS   = 5
PHYS_DIM   = 2
IMAGE_SIZE = 32
N_EPOCHS   = 1000
LR         = 0.01
LOG_EVERY  = 10
SAVE_DIR   = "checkpoints"
ASSETS_DIR = "assets/mnist"
DEVICE     = "cpu"  # change to "cuda" for GPU

torch.manual_seed(42)

## Helper Functions

In [ ]:
def load_mnist_image(idx=0, size=32):
    """Load the idx-th MNIST image and resize to size x size."""
    transform = transforms.Compose([
        transforms.Resize((size, size)),
        transforms.ToTensor(),
    ])
    dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    img, label = dataset[idx]
    return img[0], label


def init_model1_from_image(graph, image_tensor, backend):
    """Initialize QCTN with shared core 'A' from image data."""
    qctn = QCTN(graph, backend=backend)
    core_name = 'A'
    if core_name not in qctn.cores_weights:
        return qctn
    shape = qctn.cores_weights[core_name].shape
    total = 1
    for d in shape:
        total *= d
    img_flat = image_tensor.flatten()
    if img_flat.numel() < total:
        repeats = (total // img_flat.numel()) + 1
        img_flat = img_flat.repeat(repeats)[:total]
    else:
        img_flat = img_flat[:total]
    core_data = img_flat.reshape(shape).to(dtype=torch.complex64)
    core_data = core_data / torch.norm(core_data)
    qctn.cores_weights[core_name] = backend.convert_to_tensor(core_data)
    return qctn


def tensor_to_image(result_np, size=32):
    """Convert a flattened tensor result into a normalized image array."""
    if np.iscomplexobj(result_np):
        result_np = np.abs(result_np)
    result_np = result_np.flatten()
    r_min, r_max = result_np.min(), result_np.max()
    if r_max > r_min:
        result_np = (result_np - r_min) / (r_max - r_min)
    else:
        result_np = np.zeros_like(result_np)
    total = size * size
    if result_np.size < total:
        result_np = np.pad(result_np, (0, total - result_np.size))
    else:
        result_np = result_np[:total]
    return (result_np.reshape(size, size) * 255).astype(np.uint8)

## Model 1: Target (from MNIST image)

In [ ]:
backend = BackendFactory.create_backend('pytorch', device=DEVICE, dtype='float32')
engine  = EngineCommon(backend=backend, strategy_mode="full")

graph1 = "\n".join(["-2-A-2-"] * N_QUBITS)
mnist_image, mnist_label = load_mnist_image(idx=0, size=IMAGE_SIZE)
print(f"MNIST label: {mnist_label}, image shape: {mnist_image.shape}")

model1 = init_model1_from_image(graph1, mnist_image, backend)

# Show original MNIST image
fig, ax = plt.subplots(figsize=(3, 3))
ax.imshow(mnist_image.numpy(), cmap='gray')
ax.set_title(f'MNIST digit {mnist_label}')
ax.axis('off')
plt.show()

## Model 2: Trainable

In [ ]:
graph2 = "\n".join(["-2-A-2-"] * N_QUBITS)
model2 = QCTN(graph2, backend=backend)
model2.auto_init()

for c in model2.cores:
    core = model2.cores_weights[c]
    noise = torch.randn_like(core.tensor) * 0.01
    core.set(core.tensor + noise, core.scale)
    core.requires_grad_(True)

print(f"Model 2: {model2.ncores} cores, {len(model2.parameters())} trainable")
for c in model2.cores:
    t = model2.cores_weights[c]
    print(f"  Core '{c}': shape={tuple(t.shape)}")

## Training

In [ ]:
optimizer = create_optimizer("adam", model2.parameters(), backend=backend, lr=LR)
loss_history = []

for step in range(1, N_EPOCHS + 1):
    loss_val, grads = engine.contract_for_gradient(model2, target=model1, loss='mse')
    optimizer.step(list(grads))
    lv = float(loss_val)
    loss_history.append(lv)
    if step % LOG_EVERY == 0 or step == 1:
        print(f"  Step {step:4d}/{N_EPOCHS}  loss={lv:.6f}")

print(f"\nInitial loss: {loss_history[0]:.6f}")
print(f"Final   loss: {loss_history[-1]:.6f}")
print(f"Loss reduced: {loss_history[0] - loss_history[-1]:.6f}")

## Save Model

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ASSETS_DIR, exist_ok=True)

save_path = os.path.join(SAVE_DIR, "mnist_model2.safetensors")
model2.save_cores(save_path, metadata={
    'mnist_label': str(mnist_label),
    'n_epochs': str(N_EPOCHS),
    'final_loss': f"{loss_history[-1]:.6f}",
})
print(f"Model saved: {save_path}")

## Validation: Image Comparison

In [ ]:
with torch.no_grad():
    result1 = engine.contract(model1)
    result2 = engine.contract(model2)

img1_np = tensor_to_image(backend.tensor_to_numpy(result1), IMAGE_SIZE)
img2_np = tensor_to_image(backend.tensor_to_numpy(result2), IMAGE_SIZE)

# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img1_np, cmap='gray')
axes[0].set_title('Target (Model 1)')
axes[0].axis('off')

axes[1].imshow(img2_np, cmap='gray')
axes[1].set_title('Learned (Model 2)')
axes[1].axis('off')

diff = np.abs(img1_np.astype(float) - img2_np.astype(float))
axes[2].imshow(diff, cmap='hot')
axes[2].set_title(f'|Difference| (MSE={np.mean(diff**2):.1f})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

# Save images
Image.fromarray(img1_np, mode="L").save(os.path.join(ASSETS_DIR, "model1_target.png"))
Image.fromarray(img2_np, mode="L").save(os.path.join(ASSETS_DIR, "model2_output.png"))
print("Images saved to", ASSETS_DIR)

## Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history)
ax.set_xlabel('Step')
ax.set_ylabel('MSE Loss')
ax.set_title(f'Training Loss (MNIST digit {mnist_label})')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Reload Validation

In [ ]:
model2_loaded = QCTN(graph2, backend=backend)
model2_loaded.auto_init()
model2_loaded.load_cores(save_path)

with torch.no_grad():
    result_loaded = engine.contract(model2_loaded)

result2_np = backend.tensor_to_numpy(result2)
result_loaded_np = backend.tensor_to_numpy(result_loaded)
reload_err = np.max(np.abs(result2_np - result_loaded_np))
print(f"Max reload error: {reload_err:.2e}")